# 10c — Gene Regulatory Network Visualization

**Layout:** TF (diamond, center) → Enhancers (squares, inner ring) → Target genes (circles, outer ring)

**Triplet selection:** ≤3 enhancers per target gene (top-3 by score), then top-20 human-up triplets per TF.
Genes are globally spread around the outer ring to avoid label overlap.

**TF selection (data-driven):** TFs with `summed_score ≥ MIN_TF_SCORE` AND `median(tf_baseMean) ≥ MIN_TF_EXPR`.
No fixed count — number varies by cell type.

**Outputs per cell type** (in `output/grn_viz/{CellType}/`):
- `{TF}.pdf` — individual TF, 1×3 panels, gene labels
- `{TF}_nolabels.pdf` — same, no gene labels
- `combined.pdf` — all TFs together, gene labels
- `combined_nolabels.pdf` — same, no gene labels

**Three coloring panels per figure:**
1. **Node type** — TF=purple · Enhancer=orange · Gene=green
2. **Continuous** — `-log10(padj) × sign(lfc)` (Div_Human_vs_Apes)
3. **Binary** — evolutionary branch (ultra-robust):
   🔴 Human vs Apes · 🔵 Apes vs Monkeys · 🟢 Old World vs New World · ⬜ n.s.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
import matplotlib.patheffects
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family':      'sans-serif',
    'font.sans-serif':  ['Helvetica', 'Arial', 'DejaVu Sans'],
    'pdf.fonttype':     42,   # embed fonts in PDF (no Type 3)
    'ps.fonttype':      42,
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
})

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
BASE     = Path('/mp/nfs4krb5/bs-nas02/bsse_group_treutlein/USERS/jjans/analysis/adult_intestine')
TRIPLETS = BASE / 'peaks/cross_species_consensus_v3/22_publication_regulatory_panels/tables'
DA_DIR   = BASE / 'peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk'
RNA_DIR  = BASE / 'rna/pseudobulk_deseq2/rna_differential_results'
OUT_DIR  = BASE / 'peaks/peak_calling/atac_pipeline/output/grn_viz'

# ── Triplet / TF selection parameters ─────────────────────────────────────
TOP_N           = 20
TOP_N_COMBINED  = 8
MAX_ENH_GENE    = 3
MIN_TF_SCORE    = 500
MIN_TF_EXPR     = 2

# ── Differential analysis thresholds ──────────────────────────────────────
RNA_PADJ_THR = 0.05
RNA_LFC_THR  = 1.0
CONT_CLAMP   = 10.0

# ── Individual TF layout radii ─────────────────────────────────────────────
R_ENH      = 1.5
R_GENE     = 2.0    # tight — enh→gene gap ~0.3 units
R_TF_NODE  = 0.38
R_E        = 0.13
R_G        = 0.16
ARC_F      = 0.85
GENE_GAP   = 0.10

# ── Combined circular layout radii ────────────────────────────────────────
RC_TF_RING = 1.1
RC_ENH     = 2.2
RC_GENE    = 3.2    # reduced proportionally from 4.2
RC_TF_NODE = 0.40
RC_E       = 0.12
RC_G       = 0.14
RC_GENE_GAP= 0.055

# ── Force-directed layout scale ────────────────────────────────────────────
FORCE_SCALE = 3.8
FTF_NODE    = 0.28
FE_SIZE     = 0.11
FG_SIZE     = 0.11

# ── Node type colours ──────────────────────────────────────────────────────
COL_TF   = '#5B4A9C'
COL_ENH  = '#E07B39'
COL_GENE = '#2E8B57'
GREY     = '#C8C8C8'

In [ ]:
# ── Contrast definitions ───────────────────────────────────────────────────
CONTRASTS = [
    {'da_file': 'Div_Human_vs_Apes_DA.csv',
     'rna_file': 'Div_Human_vs_Apes_RNA_DE.csv',
     'color': '#D62728', 'label': 'Human vs Apes',         'priority': 1},
    {'da_file': 'Node_Apes_vs_Monkeys_DA.csv',
     'rna_file': 'Node_Apes_vs_Monkeys_RNA_DE.csv',
     'color': '#1F77B4', 'label': 'Apes vs Monkeys',       'priority': 2},
    {'da_file': 'Node4_OldWorld_vs_Marmoset_DA.csv',
     'rna_file': 'Node4_OldWorld_vs_Marmoset_RNA_DE.csv',
     'color': '#2CA02C', 'label': 'Old World vs New World', 'priority': 3},
]

# Cell types to process
CELL_TYPES = [
    'Enterocytes', 'Macrophages', 'T.cells', 'Naive.B.cells', 'Plasma.B.cells'
]

In [ ]:
# ── Data loading ───────────────────────────────────────────────────────────

def select_tfs(cell_type):
    """Select TFs by score+expression thresholds (no fixed count)."""
    df  = pd.read_csv(TRIPLETS / f'{cell_type}__triplet_ranking.tsv', sep='\t')
    hu  = df[df['direction'] == 'Human up'].copy()
    hu  = hu.sort_values('total_score_abs', ascending=False)
    hu  = hu.groupby('Gene', group_keys=False).head(MAX_ENH_GENE)

    rows = []
    for tf, grp in hu.groupby('TF'):
        score = grp.nlargest(TOP_N, 'total_score_abs')['total_score_abs'].sum()
        expr  = grp['tf_baseMean'].median()
        if score >= MIN_TF_SCORE and expr >= MIN_TF_EXPR:
            rows.append((tf, score))
    rows.sort(key=lambda x: -x[1])
    return [tf for tf, _ in rows]


def load_triplets(cell_type, tf, top_n=TOP_N):
    """Human-up triplets: ≤MAX_ENH_GENE per gene, then top top_n by score."""
    df  = pd.read_csv(TRIPLETS / f'{cell_type}__triplet_ranking.tsv', sep='\t')
    sub = df[(df['TF'] == tf) & (df['direction'] == 'Human up')]
    sub = sub.sort_values('total_score_abs', ascending=False)
    sub = sub.groupby('Gene', group_keys=False).head(MAX_ENH_GENE)
    return sub.nlargest(top_n, 'total_score_abs').reset_index(drop=True)


def load_da_peaks(cell_type, da_file):
    p = DA_DIR / cell_type / 'atac_ultra_robust' / da_file
    return set() if not p.exists() else set(pd.read_csv(p)['peak_id'])


def load_rna_genes(cell_type, rna_file):
    p = RNA_DIR / cell_type / rna_file
    if not p.exists():
        return set()
    df  = pd.read_csv(p)
    return set(df[(df['padj'] < RNA_PADJ_THR) & (df['log2FoldChange'] > RNA_LFC_THR)]['gene'])


def safe_score(lfc, padj, clamp=CONT_CLAMP):
    with np.errstate(divide='ignore', invalid='ignore'):
        s = -np.log10(np.clip(float(padj), 1e-300, 1.0)) * np.sign(float(lfc))
    return float(np.clip(s, -clamp, clamp))


# Print selected TFs for reference
print(f'TF selection  (score≥{MIN_TF_SCORE}, expr≥{MIN_TF_EXPR}):')
for ct in CELL_TYPES:
    tfs = select_tfs(ct)
    print(f'  {ct} ({len(tfs)} TFs): {tfs}')

In [ ]:
# ── Angle spreading ────────────────────────────────────────────────────────

def _spread_angles(angles, min_gap_rad=0.07):
    """
    Spread a sorted list of angles so adjacent pairs are ≥ min_gap_rad apart.
    Falls back to uniform distribution if there are too many nodes to fit.
    """
    n = len(angles)
    if n <= 1:
        return list(angles)
    if n * min_gap_rad >= 2 * np.pi * 0.98:
        base = angles[0]
        return [base + 2 * np.pi * i / n for i in range(n)]
    a = list(angles)
    for _ in range(300):
        changed = False
        for i in range(1, n):
            if a[i] - a[i - 1] < min_gap_rad:
                mid = (a[i - 1] + a[i]) / 2
                a[i - 1] = mid - min_gap_rad / 2
                a[i]     = mid + min_gap_rad / 2
                changed  = True
        if not changed:
            break
    return a

In [ ]:
# ── Individual TF layout ───────────────────────────────────────────────────

def compute_layout(triplets):
    """
    Enhancers: evenly spaced on inner ring.
    Genes: initially placed in arc behind their primary enhancer,
    then globally spread with min-gap to eliminate label overlap.
    """
    pids  = list(triplets['peak_id'].unique())
    n_enh = len(pids)
    enh_angle = {pid: np.pi / 2 + 2 * np.pi * i / n_enh
                 for i, pid in enumerate(pids)}
    enh_pos   = {pid: (R_ENH * np.cos(a), R_ENH * np.sin(a))
                 for pid, a in enh_angle.items()}

    gene_primary = (triplets
                    .sort_values('total_score_abs', ascending=False)
                    .groupby('Gene')['peak_id'].first().to_dict())
    enh_to_genes = {}
    for gene, pid in gene_primary.items():
        enh_to_genes.setdefault(pid, []).append(gene)

    arc_step = 2 * np.pi / n_enh
    raw_gene_angle = {}
    for pid, genes in enh_to_genes.items():
        base = enh_angle[pid]
        n    = len(genes)
        offs = np.linspace(-ARC_F * arc_step / 2,
                            ARC_F * arc_step / 2, n) if n > 1 else [0.0]
        for gene, off in zip(genes, offs):
            raw_gene_angle[gene] = base + off

    genes_sorted  = sorted(raw_gene_angle, key=lambda g: raw_gene_angle[g])
    gene_angles   = _spread_angles([raw_gene_angle[g] for g in genes_sorted],
                                    min_gap_rad=GENE_GAP)
    gene_pos = {g: (R_GENE * np.cos(a), R_GENE * np.sin(a))
                for g, a in zip(genes_sorted, gene_angles)}

    return enh_pos, gene_pos

In [ ]:
# ── Combined (multi-TF) layout ─────────────────────────────────────────────

def compute_combined_layout(all_triplets_dict):
    """
    TFs on inner ring.
    Enhancer angle = weighted mean of its TF angles (shared enhancers cluster
    BETWEEN their TFs, showing co-regulation visually).
    Genes placed at mean enhancer angle, then globally spread.
    """
    tfs    = list(all_triplets_dict.keys())
    n_tf   = len(tfs)
    tfa    = {tf: np.pi / 2 + 2 * np.pi * i / n_tf
              for i, tf in enumerate(tfs)}
    tf_pos = {tf: (RC_TF_RING * np.cos(a), RC_TF_RING * np.sin(a))
              for tf, a in tfa.items()}

    all_t  = pd.concat(list(all_triplets_dict.values()), ignore_index=True)

    # Enhancer angles: weighted mean over TFs that regulate them
    era = {}
    for pid, grp in all_t.groupby('peak_id'):
        tfs_h  = [tf for tf in grp['TF'].unique() if tf in tfa]
        scores = [grp[grp['TF'] == tf]['total_score_abs'].sum() for tf in tfs_h]
        total  = sum(scores) or 1.0
        sin_w  = sum(s * np.sin(tfa[tf]) for tf, s in zip(tfs_h, scores)) / total
        cos_w  = sum(s * np.cos(tfa[tf]) for tf, s in zip(tfs_h, scores)) / total
        era[pid] = np.arctan2(sin_w, cos_w)

    ps_sorted = sorted(era, key=lambda p: era[p])
    enh_angles = dict(zip(ps_sorted,
                          _spread_angles([era[p] for p in ps_sorted],
                                          min_gap_rad=0.07)))
    enh_pos = {pid: (RC_ENH * np.cos(a), RC_ENH * np.sin(a))
               for pid, a in enh_angles.items()}

    # Gene angles: mean of their enhancers
    g2p = {}
    for pid, grp in all_t.groupby('peak_id'):
        for gene in grp['Gene'].unique():
            g2p.setdefault(gene, []).append(pid)

    gra = {}
    for gene, pids in g2p.items():
        angles = [enh_angles[p] for p in pids if p in enh_angles]
        if angles:
            gra[gene] = np.arctan2(np.mean(np.sin(angles)),
                                    np.mean(np.cos(angles)))

    gs_sorted  = sorted(gra, key=lambda g: gra[g])
    gene_pos = {g: (RC_GENE * np.cos(a), RC_GENE * np.sin(a))
                for g, a in zip(gs_sorted,
                                _spread_angles([gra[g] for g in gs_sorted],
                                                min_gap_rad=RC_GENE_GAP))}

    return tf_pos, enh_pos, gene_pos

In [ ]:
# ── Force-directed layout ─────────────────────────────────────────────────

def compute_force_layout(all_triplets_dict, k=1.4, seed=42):
    """
    Spring-force layout seeded from hierarchy (TFs→inner, enh→mid, genes→outer).
    Returns positions scaled to [-FORCE_SCALE, FORCE_SCALE].
    """
    all_t = pd.concat(list(all_triplets_dict.values()), ignore_index=True)
    tfs   = list(all_triplets_dict.keys())
    peaks = list(all_t['peak_id'].unique())
    genes = list(all_t['Gene'].unique())

    G = nx.Graph()
    for tf in tfs:   G.add_node(tf,  ntype='tf')
    for p  in peaks: G.add_node(p,   ntype='enh')
    for g  in genes: G.add_node(g,   ntype='gene')
    for _, row in all_t.drop_duplicates(['TF', 'peak_id']).iterrows():
        G.add_edge(row['TF'], row['peak_id'],
                   weight=float(row['total_score_abs']))
    for _, row in all_t.drop_duplicates(['peak_id', 'Gene']).iterrows():
        G.add_edge(row['peak_id'], row['Gene'],
                   weight=float(row['total_score_abs']))

    # Hierarchical seed: TFs at r≈0.5, enh at r≈2, genes at r≈4
    n_tf, n_p, n_g = len(tfs), len(peaks), len(genes)
    init = {}
    for i, tf in enumerate(tfs):
        a = 2*np.pi*i/n_tf
        init[tf] = np.array([0.5*np.cos(a), 0.5*np.sin(a)])
    for i, p in enumerate(peaks):
        a = 2*np.pi*i/n_p
        init[p]  = np.array([2.0*np.cos(a), 2.0*np.sin(a)])
    for i, g in enumerate(genes):
        a = 2*np.pi*i/n_g
        init[g]  = np.array([4.0*np.cos(a), 4.0*np.sin(a)])

    pos = nx.spring_layout(G, pos=init, k=k, iterations=120, seed=seed,
                           weight='weight')

    # Normalise to [-1,1] then scale
    coords = np.array(list(pos.values()))
    cx, cy = coords.mean(0)
    rng    = max(coords[:,0].max()-coords[:,0].min(),
                 coords[:,1].max()-coords[:,1].min()) / 2 or 1.0
    scale  = FORCE_SCALE / rng
    pos_s  = {n: np.array([(pos[n][0]-cx)*scale, (pos[n][1]-cy)*scale])
              for n in pos}

    return ({tf: pos_s[tf] for tf in tfs},
            {p:  pos_s[p]  for p  in peaks},
            {g:  pos_s[g]  for g  in genes},
            all_t)


def draw_force(ax, all_t, tf_pos, enh_pos, gene_pos,
               tf_colors, enh_colors, gene_colors,
               title='', show_gene_labels=True):
    """Draw force-directed GRN on ax."""
    ax.set_aspect('equal')
    ax.axis('off')
    all_xy = (list(tf_pos.values()) + list(enh_pos.values())
              + list(gene_pos.values()))
    xs = [float(p[0]) for p in all_xy]
    ys = [float(p[1]) for p in all_xy]
    pad = FTF_NODE * 4
    ax.set_xlim(min(xs) - pad, max(xs) + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + pad)
    mx = float(all_t['total_score_abs'].max()) or 1.0

    # TF → Enhancer edges
    for _, row in all_t.drop_duplicates(['TF', 'peak_id']).iterrows():
        tf = row['TF']; pid = row['peak_id']
        if tf not in tf_pos or pid not in enh_pos: continue
        tp = tf_pos[tf]; ep = enh_pos[pid]
        ax.plot([float(tp[0]), float(ep[0])],
                [float(tp[1]), float(ep[1])],
                color='#888', lw=0.5, alpha=0.22, zorder=1)

    # Enhancer → Gene edges
    for _, row in all_t.drop_duplicates(['peak_id', 'Gene']).iterrows():
        pid = row['peak_id']; gene = row['Gene']
        if pid not in enh_pos or gene not in gene_pos: continue
        ep = enh_pos[pid]; gp = gene_pos[gene]
        ax.plot([float(ep[0]), float(gp[0])],
                [float(ep[1]), float(gp[1])],
                color='#aaa',
                lw=0.25 + 0.8*(float(row['total_score_abs'])/mx),
                alpha=0.20, zorder=1)

    # Enhancer squares
    for pid, p in enh_pos.items():
        ax.add_patch(_square(float(p[0]), float(p[1]), FE_SIZE,
                             facecolor=enh_colors.get(pid, GREY),
                             edgecolor='white', linewidth=0.35, zorder=3))

    # Gene circles + labels
    for gene, p in gene_pos.items():
        gx, gy = float(p[0]), float(p[1])
        ax.add_patch(mpatches.Circle((gx, gy), FG_SIZE,
                     facecolor=gene_colors.get(gene, GREY),
                     edgecolor='white', linewidth=0.35, zorder=3))
        if show_gene_labels:
            ax.text(gx, gy - FG_SIZE - 0.06, gene,
                    ha='center', va='top', fontsize=4.5,
                    zorder=5, color='#222')

    # TF diamonds (top layer)
    for tf, p in tf_pos.items():
        tx, ty = float(p[0]), float(p[1])
        col = tf_colors.get(tf, GREY) if isinstance(tf_colors, dict) else tf_colors
        ax.add_patch(_diamond(tx, ty, FTF_NODE,
                              facecolor=col, edgecolor='white',
                              linewidth=0.9, zorder=5))
        ax.text(tx, ty, tf, ha='center', va='center',
                fontsize=5.5, fontweight='bold', color='white', zorder=6,
                path_effects=[matplotlib.patheffects.withStroke(
                    linewidth=1.3, foreground='#333')])
    if title:
        ax.set_title(title, fontsize=9, pad=5)

In [ ]:
# ── Kamada-Kawai layout ────────────────────────────────────────────────────

def compute_kamada_layout(all_triplets_dict):
    """
    KK on a simplified TF→Gene graph (no enhancer nodes).
    Gives a clean organic layout; enhancers are then placed at the
    score-weighted centroid of their connected TF and gene positions.
    """
    all_t = pd.concat(list(all_triplets_dict.values()), ignore_index=True)
    tfs   = list(all_triplets_dict.keys())
    peaks = list(all_t['peak_id'].unique())
    genes = list(all_t['Gene'].unique())

    # Simplified TF-Gene graph for KK (much smaller, cleaner result)
    G2 = nx.Graph()
    for tf   in tfs:   G2.add_node(tf,   ntype='tf')
    for gene in genes: G2.add_node(gene, ntype='gene')
    for (tf, gene), sc in (all_t.groupby(['TF', 'Gene'])['total_score_abs']
                               .sum().items()):
        G2.add_edge(tf, gene, weight=float(sc))

    pos2   = nx.kamada_kawai_layout(G2, weight='weight')
    coords = np.array(list(pos2.values()))
    cx, cy = coords.mean(0)
    rng2   = max(coords[:, 0].max() - coords[:, 0].min(),
                 coords[:, 1].max() - coords[:, 1].min()) / 2 or 1.0
    sc     = FORCE_SCALE / rng2
    pos_s  = {n: np.array([(pos2[n][0] - cx) * sc,
                            (pos2[n][1] - cy) * sc]) for n in pos2}

    # Place each enhancer at score-weighted centroid of its TFs + genes
    rng_jit = np.random.default_rng(42)
    enh_pos = {}
    for pid in peaks:
        rows = all_t[all_t['peak_id'] == pid]
        xs, ys, ws = [], [], []
        for _, row in rows.iterrows():
            for node in (row['TF'], row['Gene']):
                if node in pos_s:
                    xs.append(float(pos_s[node][0]))
                    ys.append(float(pos_s[node][1]))
                    ws.append(float(row['total_score_abs']))
        if ws:
            w_sum = sum(ws)
            px = sum(x * w for x, w in zip(xs, ws)) / w_sum
            py = sum(y * w for y, w in zip(ys, ws)) / w_sum
        else:
            px, py = 0.0, 0.0
        enh_pos[pid] = np.array([px, py]) + rng_jit.normal(0, 0.12, 2)

    return ({tf: pos_s[tf] for tf in tfs},
            enh_pos,
            {g:  pos_s[g]  for g  in genes},
            all_t)


# ── Constrained spring layout ──────────────────────────────────────────────

def compute_constrained_layout(all_triplets_dict, k=2.5, seed=42):
    """
    Spring layout with TF positions fixed on a ring.
    Enhancers and genes settle freely around the TFs they connect to,
    revealing co-regulation clusters without the full hairball problem.
    """
    all_t = pd.concat(list(all_triplets_dict.values()), ignore_index=True)
    tfs   = list(all_triplets_dict.keys())
    peaks = list(all_t['peak_id'].unique())
    genes = list(all_t['Gene'].unique())

    G = nx.Graph()
    for tf in tfs:   G.add_node(tf,  ntype='tf')
    for p  in peaks: G.add_node(p,   ntype='enh')
    for g  in genes: G.add_node(g,   ntype='gene')
    for _, row in all_t.drop_duplicates(['TF', 'peak_id']).iterrows():
        G.add_edge(row['TF'], row['peak_id'],
                   weight=float(row['total_score_abs']))
    for _, row in all_t.drop_duplicates(['peak_id', 'Gene']).iterrows():
        G.add_edge(row['peak_id'], row['Gene'],
                   weight=float(row['total_score_abs']))

    n_tf = len(tfs)
    init = {}
    for i, tf in enumerate(tfs):
        a = np.pi / 2 + 2 * np.pi * i / n_tf
        init[tf] = np.array([np.cos(a), np.sin(a)])
    for i, p in enumerate(peaks):
        a = 2 * np.pi * i / len(peaks)
        init[p] = np.array([2.5 * np.cos(a), 2.5 * np.sin(a)])
    for i, g in enumerate(genes):
        a = 2 * np.pi * i / len(genes)
        init[g] = np.array([4.0 * np.cos(a), 4.0 * np.sin(a)])

    pos = nx.spring_layout(G, pos=init, fixed=list(tfs),
                           k=k, iterations=300, seed=seed, weight='weight')

    coords = np.array(list(pos.values()))
    cx, cy = coords.mean(0)
    rng2   = max(coords[:, 0].max() - coords[:, 0].min(),
                 coords[:, 1].max() - coords[:, 1].min()) / 2 or 1.0
    sc     = FORCE_SCALE / rng2
    pos_s  = {n: np.array([(pos[n][0] - cx) * sc,
                            (pos[n][1] - cy) * sc]) for n in pos}

    return ({tf: pos_s[tf] for tf in tfs},
            {p:  pos_s[p]  for p  in peaks},
            {g:  pos_s[g]  for g  in genes},
            all_t)

In [ ]:
# ── Tripartite column layout ───────────────────────────────────────────────

from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch

TRIP_X_TF    = -5.0
TRIP_X_ENH   =  0.0
TRIP_X_GENE  =  5.0
TRIP_ENH_GAP =  0.22   # y-gap between enhancer rows
TRIP_GENE_GAP = 0.22   # y-gap between gene rows


def compute_tripartite_layout(all_triplets_dict):
    """
    Three-column layout sorted to minimise edge crossings:
    • TFs (left)  sorted by total score descending
    • Enhancers (middle)  sorted by primary TF rank then score
    • Genes (right)  sorted by primary enhancer rank then name
    TF y-position = centroid of its enhancers' y-positions.
    Returns (tf_pos, enh_pos, gene_pos, all_t, y_span) where
    y_span is the data-coordinate height (for figsize computation).
    """
    all_t = pd.concat(list(all_triplets_dict.values()), ignore_index=True)
    tfs   = list(all_triplets_dict.keys())
    peaks = list(all_t['peak_id'].unique())
    genes = list(all_t['Gene'].unique())

    # ── sort TFs by score ─────────────────────────────────────────────────
    tf_score = all_t.groupby('TF')['total_score_abs'].sum().to_dict()
    tfs_ord  = sorted(tfs, key=lambda t: -tf_score.get(t, 0))
    tf_rank  = {tf: i for i, tf in enumerate(tfs_ord)}

    # ── primary TF per enhancer ───────────────────────────────────────────
    enh_primary = (all_t.groupby('peak_id')
                        .apply(lambda g:
                               g.groupby('TF')['total_score_abs'].sum().idxmax())
                        .to_dict())
    enh_score = all_t.groupby('peak_id')['total_score_abs'].sum().to_dict()
    enh_ord   = sorted(peaks, key=lambda p: (
        tf_rank.get(enh_primary.get(p, tfs_ord[0]), 0),
        -enh_score.get(p, 0)
    ))

    n_enh = len(enh_ord)
    enh_y = {p: (n_enh - 1) / 2 * TRIP_ENH_GAP - i * TRIP_ENH_GAP
             for i, p in enumerate(enh_ord)}
    enh_rank = {p: i for i, p in enumerate(enh_ord)}

    # TF y = centroid of its enhancers
    tf_ys = {tf: [] for tf in tfs_ord}
    for p in enh_ord:
        tf = enh_primary.get(p, tfs_ord[0])
        if tf in tf_ys:
            tf_ys[tf].append(enh_y[p])
    tf_y = {tf: (float(np.mean(ys)) if ys else 0.0) for tf, ys in tf_ys.items()}

    # ── primary enhancer per gene ─────────────────────────────────────────
    gene_primary = (all_t.groupby('Gene')
                         .apply(lambda g:
                                g.groupby('peak_id')['total_score_abs'].sum().idxmax())
                         .to_dict())
    genes_ord = sorted(genes, key=lambda g: (
        enh_rank.get(gene_primary.get(g, enh_ord[0]), 0), g
    ))
    n_gene = len(genes_ord)
    gene_y = {g: (n_gene - 1) / 2 * TRIP_GENE_GAP - i * TRIP_GENE_GAP
              for i, g in enumerate(genes_ord)}

    tf_pos   = {tf: np.array([TRIP_X_TF,   tf_y[tf]])  for tf in tfs}
    enh_pos  = {p:  np.array([TRIP_X_ENH,  enh_y[p]])  for p  in peaks}
    gene_pos = {g:  np.array([TRIP_X_GENE, gene_y[g]]) for g  in genes}

    y_span = (n_enh - 1) * TRIP_ENH_GAP + 1.0
    return tf_pos, enh_pos, gene_pos, all_t, y_span


def _bezier(ax, x0, y0, x1, y1, color, lw, alpha):
    """Cubic bezier S-curve between two points."""
    dx = abs(x1 - x0) * 0.45
    path = MPath(
        [(x0, y0), (x0 + dx, y0), (x1 - dx, y1), (x1, y1)],
        [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4]
    )
    ax.add_patch(PathPatch(path, facecolor='none', edgecolor=color,
                           linewidth=lw, alpha=alpha, zorder=1))


def draw_tripartite(ax, all_t, tf_pos, enh_pos, gene_pos,
                    tf_colors, enh_colors, gene_colors,
                    title='', show_gene_labels=True):
    """Draw three-column GRN with bezier-curve edges."""
    ax.set_aspect('auto')
    ax.axis('off')

    all_ys = ([float(p[1]) for p in tf_pos.values()]
              + [float(p[1]) for p in enh_pos.values()]
              + [float(p[1]) for p in gene_pos.values()])
    y_pad = 0.8
    ax.set_ylim(min(all_ys) - y_pad, max(all_ys) + y_pad)
    ax.set_xlim(TRIP_X_TF - 2.5, TRIP_X_GENE + 2.5)

    mx = float(all_t['total_score_abs'].max()) or 1.0

    # Precompute edge weights
    te_w = all_t.groupby(['TF', 'peak_id'])['total_score_abs'].sum().to_dict()
    eg_w = all_t.groupby(['peak_id', 'Gene'])['total_score_abs'].max().to_dict()

    # TF → Enhancer bezier edges
    for _, row in all_t.drop_duplicates(['TF', 'peak_id']).iterrows():
        tf = row['TF']; pid = row['peak_id']
        if tf not in tf_pos or pid not in enh_pos: continue
        w = 0.15 + 0.65 * te_w.get((tf, pid), 0) / mx
        tp = tf_pos[tf]; ep = enh_pos[pid]
        _bezier(ax, float(tp[0]), float(tp[1]),
                    float(ep[0]), float(ep[1]), '#999', w, 0.28)

    # Enhancer → Gene bezier edges
    for _, row in all_t.drop_duplicates(['peak_id', 'Gene']).iterrows():
        pid = row['peak_id']; gene = row['Gene']
        if pid not in enh_pos or gene not in gene_pos: continue
        w = 0.15 + 0.65 * eg_w.get((pid, gene), 0) / mx
        ep = enh_pos[pid]; gp = gene_pos[gene]
        _bezier(ax, float(ep[0]), float(ep[1]),
                    float(gp[0]), float(gp[1]), '#bbb', w, 0.25)

    # Enhancer squares (middle column)
    enh_sz = 0.08
    for pid, p in enh_pos.items():
        ax.add_patch(_square(float(p[0]), float(p[1]), enh_sz,
                             facecolor=enh_colors.get(pid, GREY),
                             edgecolor='white', linewidth=0.3, zorder=3))

    # Gene circles (right column) + labels
    gene_sz = 0.07
    for gene, p in gene_pos.items():
        gx, gy = float(p[0]), float(p[1])
        ax.add_patch(mpatches.Circle((gx, gy), gene_sz,
                     facecolor=gene_colors.get(gene, GREY),
                     edgecolor='white', linewidth=0.3, zorder=3))
        if show_gene_labels:
            ax.text(gx + gene_sz + 0.15, gy, gene,
                    ha='left', va='center', fontsize=5.0, zorder=5, color='#222')

    # TF diamonds (left column) + left-side labels
    tf_sz = 0.30
    for tf, p in tf_pos.items():
        tx, ty = float(p[0]), float(p[1])
        col = tf_colors.get(tf, GREY) if isinstance(tf_colors, dict) else tf_colors
        ax.add_patch(_diamond(tx, ty, tf_sz,
                              facecolor=col, edgecolor='white',
                              linewidth=0.8, zorder=5))
        ax.text(tx - tf_sz - 0.18, ty, tf,
                ha='right', va='center', fontsize=6.5, fontweight='bold',
                color=col, zorder=6)

    # Column header annotations
    y_top = max(all_ys) + y_pad * 0.65
    for x, lbl in [(TRIP_X_TF,  'Transcription\nFactors'),
                   (TRIP_X_ENH, 'Enhancers'),
                   (TRIP_X_GENE,'Target Genes')]:
        ax.text(x, y_top, lbl, ha='center', va='bottom',
                fontsize=7, color='#555', fontstyle='italic',
                multialignment='center')
    if title:
        ax.set_title(title, fontsize=9, pad=5)


def _make_trip_fn(at, tp, ep, gp):
    def fn(ax, tc, ec, gc, title='', show_gene_labels=True):
        draw_tripartite(ax, at, tp, ep, gp, tc, ec, gc,
                        title=title, show_gene_labels=show_gene_labels)
    return fn

In [ ]:
# ── Colour builders ────────────────────────────────────────────────────────

CONT_CMAP = plt.get_cmap('RdBu_r')
CONT_NORM = mcolors.TwoSlopeNorm(vmin=-CONT_CLAMP, vcenter=0, vmax=CONT_CLAMP)


def _build_contrast_sets(cell_type):
    """Pre-load DA peak sets and RNA gene sets for all contrasts."""
    peak_sets = [(c['color'], load_da_peaks(cell_type, c['da_file']))
                 for c in sorted(CONTRASTS, key=lambda x: x['priority'])]
    gene_sets = [(c['color'], load_rna_genes(cell_type, c['rna_file']))
                 for c in sorted(CONTRASTS, key=lambda x: x['priority'])]
    return peak_sets, gene_sets


def _classify_peak(pid, peak_sets):
    for col, s in peak_sets:
        if pid in s: return col
    return GREY


def _classify_gene(gene, gene_sets):
    for col, s in gene_sets:
        if gene in s: return col
    return GREY


def colors_type(triplets):
    return (COL_TF,
            {p: COL_ENH  for p in triplets['peak_id'].unique()},
            {g: COL_GENE for g in triplets['Gene'].unique()})


def colors_continuous(triplets):
    rgba = lambda s: CONT_CMAP(CONT_NORM(s))
    return (
        rgba(safe_score(triplets['tf_lfc'].mean(), triplets['tf_padj'].mean())),
        {pid: rgba(safe_score(g['da_lfc'].iloc[0], g['da_padj'].iloc[0]))
         for pid, g in triplets.groupby('peak_id', sort=False)},
        {gene: rgba(safe_score(g['target_lfc'].mean(), g['target_padj'].mean()))
         for gene, g in triplets.groupby('Gene', sort=False)}
    )


def colors_binary(triplets, peak_sets, gene_sets):
    tf = triplets['TF'].iloc[0]
    return (
        _classify_gene(tf, gene_sets),
        {p: _classify_peak(p, peak_sets)  for p in triplets['peak_id'].unique()},
        {g: _classify_gene(g, gene_sets)  for g in triplets['Gene'].unique()}
    )


# Combined variants
def colors_type_c(atd):
    all_t = pd.concat(list(atd.values()), ignore_index=True)
    return ({tf: COL_TF   for tf in atd},
            {p:  COL_ENH  for p  in all_t['peak_id'].unique()},
            {g:  COL_GENE for g  in all_t['Gene'].unique()})


def colors_continuous_c(atd):
    all_t = pd.concat(list(atd.values()), ignore_index=True)
    rgba  = lambda s: CONT_CMAP(CONT_NORM(s))
    return (
        {tf: rgba(safe_score(t['tf_lfc'].mean(), t['tf_padj'].mean()))
         for tf, t in atd.items()},
        {pid: rgba(safe_score(g['da_lfc'].iloc[0], g['da_padj'].iloc[0]))
         for pid, g in all_t.groupby('peak_id', sort=False)},
        {gene: rgba(safe_score(g['target_lfc'].mean(), g['target_padj'].mean()))
         for gene, g in all_t.groupby('Gene', sort=False)}
    )


def colors_binary_c(atd, peak_sets, gene_sets):
    all_t = pd.concat(list(atd.values()), ignore_index=True)
    return (
        {tf: _classify_gene(tf, gene_sets) for tf in atd},
        {p:  _classify_peak(p, peak_sets)  for p  in all_t['peak_id'].unique()},
        {g:  _classify_gene(g, gene_sets)  for g  in all_t['Gene'].unique()}
    )

In [ ]:
# ── Shape helpers ──────────────────────────────────────────────────────────

def _diamond(cx, cy, r, **kwargs):
    """Diamond patch: RegularPolygon with 4 vertices, orientation=0 (points at E/N/W/S)."""
    return mpatches.RegularPolygon((cx, cy), numVertices=4,
                                    radius=r, orientation=0, **kwargs)


def _square(cx, cy, r, **kwargs):
    """Axis-aligned square: RegularPolygon, orientation=π/4 (points at NE/NW/SW/SE)."""
    return mpatches.RegularPolygon((cx, cy), numVertices=4,
                                    radius=r, orientation=np.pi / 4, **kwargs)

In [ ]:
# ── Draw: individual TF ────────────────────────────────────────────────────

def draw_network(ax, triplets, enh_pos, gene_pos,
                 tf_color, enh_colors, gene_colors,
                 title='', show_gene_labels=True):
    ax.set_aspect('equal')
    ax.axis('off')
    lim = R_GENE + 1.6
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    mx = triplets['total_score_abs'].max() or 1.0

    # TF → Enhancer edges
    for pid, (ex, ey) in enh_pos.items():
        ax.plot([0, ex], [0, ey], color='#888', lw=0.5, alpha=0.30, zorder=1)

    # Enhancer → Gene edges
    for _, row in triplets.iterrows():
        pid = row['peak_id']; gene = row['Gene']
        if pid not in enh_pos or gene not in gene_pos: continue
        ex, ey = enh_pos[pid]; gx, gy = gene_pos[gene]
        ax.plot([ex, gx], [ey, gy], color='#999',
                lw=0.3 + 1.6 * (row['total_score_abs'] / mx),
                alpha=0.28, zorder=1)

    # Enhancer squares
    for pid, (ex, ey) in enh_pos.items():
        ax.add_patch(_square(ex, ey, R_E * 1.25,
                             facecolor=enh_colors.get(pid, GREY),
                             edgecolor='white', linewidth=0.4, zorder=3))

    # Gene circles + labels
    for gene, (gx, gy) in gene_pos.items():
        ax.add_patch(mpatches.Circle((gx, gy), R_G,
                     facecolor=gene_colors.get(gene, GREY),
                     edgecolor='white', linewidth=0.4, zorder=3))
        if show_gene_labels:
            adeg = np.degrees(np.arctan2(gy, gx))
            ha   = 'left' if -90 < adeg <= 90 else 'right'
            nx   = gx / R_GENE; ny = gy / R_GENE
            ax.text(gx + (R_G + 0.18) * nx, gy + (R_G + 0.18) * ny, gene,
                    ha=ha, va='center', fontsize=5.5, zorder=5, color='#222')

    # TF diamond
    tf_name = triplets['TF'].iloc[0]
    ax.add_patch(_diamond(0, 0, R_TF_NODE * 1.15,
                          facecolor=tf_color, edgecolor='white',
                          linewidth=1.0, zorder=4))
    ax.text(0, 0, tf_name, ha='center', va='center',
            fontsize=7, fontweight='bold', color='white', zorder=5,
            path_effects=[matplotlib.patheffects.withStroke(
                linewidth=1.5, foreground='#333')])
    if title:
        ax.set_title(title, fontsize=8, pad=4)

In [ ]:
# ── Draw: combined multi-TF ────────────────────────────────────────────────

def draw_combined(ax, all_triplets, tf_pos, enh_pos, gene_pos,
                  tf_colors, enh_colors, gene_colors,
                  title='', show_gene_labels=True):
    ax.set_aspect('equal')
    ax.axis('off')
    lim = RC_GENE + 1.8
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    mx = all_triplets['total_score_abs'].max() or 1.0

    # TF → Enhancer edges
    for _, row in all_triplets.drop_duplicates(['TF', 'peak_id']).iterrows():
        tf = row['TF']; pid = row['peak_id']
        if tf not in tf_pos or pid not in enh_pos: continue
        tx, ty = tf_pos[tf]; ex, ey = enh_pos[pid]
        ax.plot([tx, ex], [ty, ey], color='#888', lw=0.4, alpha=0.20, zorder=1)

    # Enhancer → Gene edges
    for _, row in all_triplets.drop_duplicates(['peak_id', 'Gene']).iterrows():
        pid = row['peak_id']; gene = row['Gene']
        if pid not in enh_pos or gene not in gene_pos: continue
        ex, ey = enh_pos[pid]; gx, gy = gene_pos[gene]
        ax.plot([ex, gx], [ey, gy], color='#aaa',
                lw=0.25 + 0.9 * (row['total_score_abs'] / mx),
                alpha=0.20, zorder=1)

    # Enhancer squares
    for pid, (ex, ey) in enh_pos.items():
        ax.add_patch(_square(ex, ey, RC_E * 1.25,
                             facecolor=enh_colors.get(pid, GREY),
                             edgecolor='white', linewidth=0.3, zorder=3))

    # Gene circles + labels
    for gene, (gx, gy) in gene_pos.items():
        ax.add_patch(mpatches.Circle((gx, gy), RC_G,
                     facecolor=gene_colors.get(gene, GREY),
                     edgecolor='white', linewidth=0.3, zorder=3))
        if show_gene_labels:
            r    = (gx**2 + gy**2)**0.5 or 1.0
            adeg = np.degrees(np.arctan2(gy, gx))
            ha   = 'left' if -90 < adeg <= 90 else 'right'
            ax.text(gx + (RC_G + 0.15) * gx / r,
                    gy + (RC_G + 0.15) * gy / r,
                    gene, ha=ha, va='center', fontsize=4.0, zorder=5, color='#222')

    # TF diamonds (drawn last — on top)
    for tf, (tx, ty) in tf_pos.items():
        col = tf_colors.get(tf, GREY) if isinstance(tf_colors, dict) else tf_colors
        ax.add_patch(_diamond(tx, ty, RC_TF_NODE * 1.15,
                              facecolor=col, edgecolor='white',
                              linewidth=0.8, zorder=5))
        ax.text(tx, ty, tf, ha='center', va='center',
                fontsize=5.0, fontweight='bold', color='white', zorder=6,
                path_effects=[matplotlib.patheffects.withStroke(
                    linewidth=1.2, foreground='#333')])
    if title:
        ax.set_title(title, fontsize=8, pad=4)

In [ ]:
# ── Legend / colorbar helpers ──────────────────────────────────────────────

def _type_handles():
    return [
        mpatches.Patch(facecolor=COL_TF,   label='TF (diamond)'),
        mpatches.Patch(facecolor=COL_ENH,  label='Enhancer (square)'),
        mpatches.Patch(facecolor=COL_GENE, label='Target gene (circle)'),
    ]


def _binary_handles():
    return ([mpatches.Patch(facecolor=c['color'], label=c['label'])
             for c in CONTRASTS]
            + [mpatches.Patch(facecolor=GREY, label='Not significant')])


def _add_colorbar(fig, left=0.36, width=0.30):
    sm  = mcm.ScalarMappable(cmap=CONT_CMAP, norm=CONT_NORM)
    sm.set_array([])
    cax = fig.add_axes([left, 0.04, width, 0.018])
    cb  = fig.colorbar(sm, cax=cax, orientation='horizontal')
    cb.set_label('-log10(padj) × sign(lfc)', fontsize=7)
    cb.ax.tick_params(labelsize=6)

In [ ]:
# ── Save helpers ──────────────────────────────────────────────────────────

def _savefig(fig, stem):
    """Write PDF (vector) and PNG (300 dpi) from the same figure."""
    fig.savefig(str(stem) + '.pdf', dpi=150, bbox_inches='tight',
                facecolor='white')
    fig.savefig(str(stem) + '.png', dpi=300, bbox_inches='tight',
                facecolor='white')


def _save_pair(path_stem, draw_fn, draw_fn_args, suptitle, figsize=(26, 9)):
    """Individual TF: 1×3 panel figure, labeled + nolabels, PDF + PNG."""
    for labeled in [True, False]:
        fig, axes = plt.subplots(1, 3, figsize=figsize,
                                 facecolor='white')
        fig.suptitle(suptitle, fontsize=9, y=0.97)
        titles = ['Node type',
                  'Continuous: −log₁₀(padj) × sign(lfc)',
                  'Binary: evolutionary branch']
        for ax, title, (tc, ec, gc) in zip(axes, titles, draw_fn_args):
            draw_fn(ax, tc, ec, gc, title=title, show_gene_labels=labeled)
        axes[0].legend(handles=_type_handles(), loc='lower left',
                       fontsize=6, frameon=True, framealpha=0.9)
        axes[2].legend(handles=_binary_handles(), loc='lower right',
                       fontsize=6, frameon=True, framealpha=0.9)
        _add_colorbar(fig)
        fig.subplots_adjust(left=0.02, right=0.98, bottom=0.08,
                            top=0.94, wspace=0.04)
        suffix = '' if labeled else '_nolabels'
        _savefig(fig, Path(str(path_stem) + suffix))
        plt.close(fig)


def _save_single(path_stem, draw_fn, tc, ec, gc,
                 panel_type, title, suptitle, figsize=(12, 11)):
    """
    Single-panel figure: labeled + nolabels, PDF + PNG.
    panel_type: 'type' | 'continuous' | 'binary'
    """
    for labeled in [True, False]:
        fig, ax = plt.subplots(1, 1, figsize=figsize, facecolor='white')
        fig.suptitle(suptitle, fontsize=10, y=0.99, va='top')
        draw_fn(ax, tc, ec, gc, title=title, show_gene_labels=labeled)

        if panel_type == 'type':
            ax.legend(handles=_type_handles(), loc='lower left',
                      fontsize=8, frameon=True, framealpha=0.9,
                      borderpad=0.8)
        elif panel_type == 'continuous':
            _add_colorbar(fig, left=0.15, width=0.70)
        elif panel_type == 'binary':
            ax.legend(handles=_binary_handles(), loc='lower right',
                      fontsize=8, frameon=True, framealpha=0.9,
                      borderpad=0.8)

        fig.subplots_adjust(left=0.03, right=0.97,
                            bottom=0.09, top=0.95)
        suffix = '' if labeled else '_nolabels'
        _savefig(fig, Path(str(path_stem) + suffix))
        plt.close(fig)

In [ ]:
# ── Main rendering loop ────────────────────────────────────────────────────

def _make_circ_fn(at, tp, ep, gp):
    def fn(ax, tc, ec, gc, title='', show_gene_labels=True):
        draw_combined(ax, at, tp, ep, gp, tc, ec, gc,
                      title=title, show_gene_labels=show_gene_labels)
    return fn

def _make_force_fn(at, tp, ep, gp):
    def fn(ax, tc, ec, gc, title='', show_gene_labels=True):
        draw_force(ax, at, tp, ep, gp, tc, ec, gc,
                   title=title, show_gene_labels=show_gene_labels)
    return fn

total_files = 0

for cell_type in CELL_TYPES:
    ct_dir  = OUT_DIR / cell_type
    ct_dir.mkdir(parents=True, exist_ok=True)
    tf_list = select_tfs(cell_type)
    print(f'\n── {cell_type}  ({len(tf_list)} TFs)')

    peak_sets, gene_sets = _build_contrast_sets(cell_type)

    # ── Individual TF plots (1×3 panel, PDF + PNG) ───────────────────────
    all_triplets_dict = {}
    for tf in tf_list:
        t = load_triplets(cell_type, tf)
        if t.empty:
            print(f'  {tf}: no triplets, skipped')
            continue
        all_triplets_dict[tf] = t

        enh_pos, gene_pos = compute_layout(t)
        tc_tp, ec_tp, gc_tp = colors_type(t)
        tc_co, ec_co, gc_co = colors_continuous(t)
        tc_bi, ec_bi, gc_bi = colors_binary(t, peak_sets, gene_sets)

        def _draw_ind(ax, tc, ec, gc, title='', show_gene_labels=True,
                      _t=t, _ep=enh_pos, _gp=gene_pos):
            draw_network(ax, _t, _ep, _gp, tc, ec, gc,
                         title=title, show_gene_labels=show_gene_labels)

        suptitle = (f'{cell_type.replace(".", " ")} — {tf}  '
                    f'(top {len(t)} Human-up triplets · ≤{MAX_ENH_GENE} enh/gene)')
        _save_pair(ct_dir / tf, _draw_ind,
                   [(tc_tp, ec_tp, gc_tp),
                    (tc_co, ec_co, gc_co),
                    (tc_bi, ec_bi, gc_bi)],
                   suptitle)
        print(f'  {tf}: {len(t)} triplets · '
              f'{t["peak_id"].nunique()} enh · {t["Gene"].nunique()} genes')
        total_files += 4

    if not all_triplets_dict:
        continue

    # ── Combined plots ────────────────────────────────────────────────────
    comb_dict = {tf: load_triplets(cell_type, tf, top_n=TOP_N_COMBINED)
                 for tf in all_triplets_dict}
    comb_dict = {tf: t for tf, t in comb_dict.items() if not t.empty}
    all_t     = pd.concat(list(comb_dict.values()), ignore_index=True)
    n_tf = len(comb_dict)
    n_e  = all_t['peak_id'].nunique()
    n_g  = all_t['Gene'].nunique()
    print(f'  combined: {n_tf} TFs · {n_e} enh · {n_g} genes')

    ct_name    = cell_type.replace('.', ' ')
    suptitle_c = f'{ct_name} — combined ({n_tf} TFs · {n_e} enh · {n_g} genes)'

    # ── Compute all five layouts ──────────────────────────────────────────
    print('    computing layouts…', end=' ', flush=True)
    tf_pos_c, enh_pos_c, gene_pos_c                  = compute_combined_layout(comb_dict)
    tf_pos_f, enh_pos_f, gene_pos_f, all_t_f         = compute_force_layout(comb_dict)
    tf_pos_k, enh_pos_k, gene_pos_k, all_t_k         = compute_kamada_layout(comb_dict)
    tf_pos_cs, enh_pos_cs, gene_pos_cs, all_t_cs     = compute_constrained_layout(comb_dict)
    tf_pos_t, enh_pos_t, gene_pos_t, all_t_t, y_span = compute_tripartite_layout(comb_dict)
    print('done')

    # ── Shared colour sets ────────────────────────────────────────────────
    tc_tp, ec_tp, gc_tp = colors_type_c(comb_dict)
    tc_co, ec_co, gc_co = colors_continuous_c(comb_dict)
    tc_bi, ec_bi, gc_bi = colors_binary_c(comb_dict, peak_sets, gene_sets)

    # ── Factory functions ─────────────────────────────────────────────────
    circ_fn   = _make_circ_fn(all_t,    tf_pos_c,  enh_pos_c,  gene_pos_c)
    force_fn  = _make_force_fn(all_t_f, tf_pos_f,  enh_pos_f,  gene_pos_f)
    kamada_fn = _make_force_fn(all_t_k, tf_pos_k,  enh_pos_k,  gene_pos_k)
    constr_fn = _make_force_fn(all_t_cs, tf_pos_cs, enh_pos_cs, gene_pos_cs)
    trip_fn   = _make_trip_fn(all_t_t,  tf_pos_t,  enh_pos_t,  gene_pos_t)

    # Tripartite figsize: height scales with number of enhancers
    x_span     = (TRIP_X_GENE + 2.5) - (TRIP_X_TF - 2.5)
    trip_h     = max(9, min(22, 16 * y_span / x_span))
    trip_fsz   = (16, trip_h)

    panels = [
        ('type',       'Node type',                         tc_tp, ec_tp, gc_tp),
        ('continuous', '−log₁₀(padj) × sign(lfc)',
                                                             tc_co, ec_co, gc_co),
        ('binary',     'Evolutionary branch (ultra-robust)', tc_bi, ec_bi, gc_bi),
    ]

    layouts = [
        ('circular',    circ_fn,   (12, 11)),
        ('spring',      force_fn,  (12, 11)),
        ('kamada',      kamada_fn, (12, 11)),
        ('constrained', constr_fn, (12, 11)),
        ('tripartite',  trip_fn,   trip_fsz),
    ]

    for layout_name, fn, fsz in layouts:
        for ptype, title, tc, ec, gc in panels:
            stem = ct_dir / f'combined_{layout_name}_{ptype}'
            _save_single(stem, fn, tc, ec, gc,
                         ptype, title, suptitle_c, figsize=fsz)
        total_files += 3 * 4  # 3 panels × 4 files each (lbl+nolbl × pdf+png)
        print(f'  {layout_name}: 12 files written')

print(f'\nDone — {total_files} files written to {OUT_DIR}')